# EXP-016 — Notebook làm việc

Hồ sơ chốt: `experiments/04_additional/EXP-016_LR_Features/experiment.ipynb`. Mặc định chỉ đọc kết quả. Bật RUN_TRAIN khi cần chạy lại; output được lưu riêng. Nhật ký duy nhất là `experiments/experiments_logs.xlsx`.

In [1]:
from pathlib import Path
import os
import pandas as pd
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/wesad/models.py").is_file())
os.chdir(ROOT)
RUN_TRAIN = False
FOLDER = ROOT / 'experiments/04_additional/EXP-016_LR_Features'
def read_csv(path, *args, **kwargs):
    path = Path(path)
    if not path.exists() and path.name == "metrics_by_subject.csv":
        path = path.with_name("results.csv")
    return pd.read_csv(path, *args, **kwargs)

# EXP-016 — LR chọn đặc trưng

Ngày 17/11/2026

EXP chung: EXP-016.

## Mục tiêu
Kiểm tra chọn đặc trưng có giúp LR tổng quát hóa sang người mới hay không.
Đối chứng là EXP-002, LR đủ 68 đặc trưng.

## Giả thuyết trước khi chạy
Giảm đặc trưng có thể giảm nhiễu, nhưng cũng có thể làm mất thông tin.
Không mặc định rằng điểm số sẽ tăng.

## Thiết lập
- Multiclass và binary; seed=42; không cân bằng lớp.
- Giữ imputer median, StandardScaler và tham số LR giống đối chứng.
- SelectKBest dùng ANOVA F; khảo sát k trong [16, 32, 48, 68].
- Inner GroupKFold 3 phần trên 14 người train để chọn k bằng Macro-F1.
- Outer LOSO test riêng một người; không sử dụng outer test để chọn k.
- R01 kiểm tra S2; R02 giữ nguyên thiết kế và chạy đủ 15 người.
- 68 là ứng viên giữ đủ đặc trưng; không bắt buộc chọn mô hình rút gọn.


In [2]:
if RUN_TRAIN:
    from pathlib import Path
    from copy import deepcopy, copy
    from datetime import datetime
    from zoneinfo import ZoneInfo
    from time import perf_counter
    import os
    import sys
    import json
    import hashlib
    import shutil
    import warnings

    ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "src/wesad/models.py").is_file()), None)
    if ROOT is None:
        raise RuntimeError("Không tìm thấy thư mục dự án.")
    os.chdir(ROOT)
    if str(ROOT / "src") not in sys.path:
        sys.path.insert(0, str(ROOT / "src"))
    os.environ.setdefault("MPLCONFIGDIR", str(Path(os.environ["TEMP"]) / "wesad_matplotlib"))

    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import yaml
    from IPython.display import display, Image
    from sklearn.pipeline import Pipeline
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import StandardScaler
    from sklearn.feature_selection import SelectKBest, f_classif
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import GroupKFold, GridSearchCV
    from sklearn.metrics import f1_score, balanced_accuracy_score, make_scorer
    from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
    from threadpoolctl import threadpool_limits


    from wesad.data import SUBJECTS, load_feature_table
    from wesad.evaluation import task_target
    from wesad.utils import read_config, file_hash, fingerprint, environment, write_json, provenance

    plt.switch_backend("Agg")
    NOTEBOOK = ROOT / "notebooks/experiments/EXP-016_working.ipynb"

    REFERENCE_DIR = ROOT / "experiments/01_baselines/EXP-002_LR"
    print("Project:", ROOT)
    print("Python:", sys.executable)


## Kiểm tra dữ liệu và đối chứng

In [3]:
if RUN_TRAIN:
    defaults = read_config(ROOT / "configs/defaults.yaml")
    FEATURES = ROOT / defaults["features"]
    SCHEMA = ROOT / defaults["schema"]
    assert file_hash(FEATURES) == defaults["feature_sha256"]
    df, feature_columns = load_feature_table(FEATURES, SCHEMA, SUBJECTS)
    assert len(df) == 6189 and len(feature_columns) == 68
    reference_manifest = json.loads((REFERENCE_DIR / "config.json").read_text(encoding="utf-8"))
    assert reference_manifest["data_sha256"] == file_hash(FEATURES)
    assert reference_manifest["environment"] == environment()
    assert reference_manifest["settings"]["balance"] is False
    shared_paths = reference_manifest["settings"]["source_paths"]
    shared_hash = fingerprint({path: hashlib.sha256((ROOT / path).read_bytes().replace(b"\r\n", b"\n")).hexdigest()
                               for path in shared_paths})
    assert shared_hash == reference_manifest["code_sha256"]
    print("PASS: checksum, 68 đặc trưng, 15 người và môi trường khớp đối chứng.")
    display(pd.crosstab(df.subject_id, df.label_original).reindex(SUBJECTS))


## Pipeline và đánh giá nested LOSO

In [4]:
if RUN_TRAIN:
    def selection_pipeline():
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
            ("scaler", StandardScaler()),
            ("selection", SelectKBest(score_func=f_classif)),
            ("model", LogisticRegression(**reference_manifest["settings"]["model_params"]["lr"])),
        ])

    def make_config(run_no, scope):
        return {
            "experiment_id": "EXP-016", "group_experiment_id": "EXP-016",
            "run_no": run_no, "run_scope": scope,
            "features": defaults["features"], "schema": defaults["schema"],
            "feature_sha256": defaults["feature_sha256"], "feature_version": defaults["feature_version"],
            "models": ["lr"], "tasks": ["multiclass", "binary"],
            "expected_subjects": list(SUBJECTS), "seed": 42,
            "balance": False, "normalization": "global_train",
            "model_params": deepcopy(reference_manifest["settings"]["model_params"]),
            "selector": "SelectKBest(f_classif)", "k_candidates": [16, 32, 48, 68],
            "nested": True, "inner_splits": 3, "scoring": "macro_f1",
            "reference_run": REFERENCE_DIR.relative_to(ROOT).as_posix(),
            "output_dir": f"outputs/experiments/EXP-016/clean_v1/{run_no}_{scope}",
        }

    def run_selection(config, subjects=None):
        code_cells = json.loads(NOTEBOOK.read_text(encoding="utf-8"))["cells"]
        notebook_hash = fingerprint(["".join(c["source"]) for c in code_cells if c["cell_type"] == "code"])
        identity = {"settings": config, "data_sha256": file_hash(FEATURES),
                    "feature_columns": feature_columns, "shared_code_sha256": shared_hash,
                    "notebook_code_sha256": notebook_hash, "environment": environment()}
        signature = fingerprint(identity)
        out = ROOT / config["output_dir"]
        out.mkdir(parents=True, exist_ok=True)
        manifest_path = out / "config.json"
        if manifest_path.exists():
            assert json.loads(manifest_path.read_text())["signature"] == signature, "Cấu hình đổi: dùng lượt chạy mới."
        write_json(manifest_path, {**provenance(), **identity, "signature": signature})
        write_json(out / "environment.json", environment())
        (out / "config.yaml").write_text(yaml.safe_dump(config, allow_unicode=True, sort_keys=False), encoding="utf-8")
        selected_subjects = list(subjects) if subjects is not None else list(SUBJECTS)
        assert selected_subjects and set(selected_subjects).issubset(SUBJECTS)
        write_json(out / "run_status.json", {"status": "running"})
        X = df[feature_columns].astype(float)
        groups = df.subject_id
        try:
            for task in config["tasks"]:
                y, labels = task_target(df, task)
                for subject in selected_subjects:
                    folder = out / "folds" / task / "lr" / subject
                    folder.mkdir(parents=True, exist_ok=True)
                    done = folder / "complete.json"
                    if done.exists():
                        receipt = json.loads(done.read_text())
                        if receipt["signature"] == signature and all((folder / name).is_file() and file_hash(folder / name) == h
                                                                     for name, h in receipt["files"].items()):
                            print(f"{task}/lr/{subject}: SKIP verified", flush=True)
                            continue
                    train = np.flatnonzero(groups.ne(subject))
                    test = np.flatnonzero(groups.eq(subject))
                    assert len(test) and set(groups.iloc[train]).isdisjoint(groups.iloc[test])
                    assert groups.iloc[train].nunique() == 14
                    X_train, y_train = X.iloc[train], y.iloc[train]
                    inner_groups = groups.iloc[train]
                    splits = list(GroupKFold(n_splits=3).split(X_train, y_train, inner_groups))
                    inner_description = []
                    for a, b in splits:
                        ga, gb = set(inner_groups.iloc[a]), set(inner_groups.iloc[b])
                        assert ga.isdisjoint(gb) and subject not in ga | gb
                        inner_description.append({"train_subjects": sorted(ga), "validation_subjects": sorted(gb)})
                    started = perf_counter()
                    with warnings.catch_warnings(record=True) as caught, threadpool_limits(limits=1):
                        warnings.simplefilter("always")
                        search = GridSearchCV(selection_pipeline(), {"selection__k": config["k_candidates"]},
                                              cv=splits, scoring=make_scorer(f1_score, average="macro", labels=labels, zero_division=0),
                                              n_jobs=1, refit=True, error_score="raise", return_train_score=False)
                        search.fit(X_train, y_train)
                        predicted = search.predict(X.iloc[test])
                    chosen = [f for f, keep in zip(feature_columns, search.best_estimator_["selection"].get_support()) if keep]
                    k = int(search.best_params_["selection__k"])
                    assert len(chosen) == k
                    prediction = df.iloc[test][["subject_id", "window_id"]].copy()
                    prediction["task"], prediction["model"] = task, "lr"
                    prediction["y_true"], prediction["y_pred"] = y.iloc[test].to_numpy(), predicted
                    metric = {"subject_id": subject, "task": task, "model": "lr", "n_test": len(test),
                              "macro_f1": float(f1_score(y.iloc[test], predicted, labels=labels, average="macro", zero_division=0)),
                              "balanced_accuracy": float(balanced_accuracy_score(y.iloc[test], predicted)),
                              "selected_k": k, "inner_macro_f1": float(search.best_score_),
                              "seconds": round(perf_counter() - started, 4)}
                    details = {"outer_train_subjects": sorted(set(groups.iloc[train])), "outer_test_subjects": [subject],
                               "inner_splits": inner_description, "best_params": {"selection__k": k},
                               "selected_features": chosen, "warnings": sorted(set(str(w.message) for w in caught)),
                               "labels": labels, "confusion_matrix": confusion_matrix(y.iloc[test], predicted, labels=labels).tolist()}
                    prediction.to_csv(folder / "predictions.csv", index=False)
                    pd.DataFrame(search.cv_results_).to_csv(folder / "inner_cv.csv", index=False)
                    write_json(folder / "metrics.json", metric)
                    write_json(folder / "details.json", details)
                    files = ["predictions.csv", "inner_cv.csv", "metrics.json", "details.json"]
                    write_json(done, {"signature": signature, "files": {f: file_hash(folder / f) for f in files}})
                    print(f"{task}/lr/{subject}: k={k}; F1={metric['macro_f1']:.4f}", flush=True)
            records, predictions, fold_details, warning_rows = [], [], {}, []
            for task in config["tasks"]:
                for subject in selected_subjects:
                    folder = out / "folds" / task / "lr" / subject
                    records.append(json.loads((folder / "metrics.json").read_text()))
                    predictions.append(read_csv(folder / "predictions.csv"))
                    detail = json.loads((folder / "details.json").read_text())
                    fold_details[f"{task}/lr/{subject}"] = detail
                    warning_rows.extend({"task": task, "subject_id": subject, "warning": w} for w in detail["warnings"])
            metrics = pd.DataFrame(records)
            metrics.to_csv(out / "metrics_by_subject.csv", index=False)
            pd.concat(predictions, ignore_index=True).to_csv(out / "predictions.csv", index=False)
            pd.DataFrame(warning_rows, columns=["task", "subject_id", "warning"]).to_csv(out / "warnings.csv", index=False)
            write_json(out / "splits.json", {key: {k: val[k] for k in ["outer_train_subjects", "outer_test_subjects", "inner_splits"]}
                                              for key, val in fold_details.items()})
            write_json(out / "selected_features.json", {key: val["selected_features"] for key, val in fold_details.items()})
            status = "complete" if set(selected_subjects) == set(SUBJECTS) else "partial"
            results = []
            for task, part in metrics.groupby("task"):
                result = {"task": task, "model": "lr", "n_subjects": len(part)}
                for name in ["macro_f1", "balanced_accuracy"]:
                    result[name + "_mean"] = float(part[name].mean())
                    result[name + "_std"] = float(part[name].std()) if len(part) > 1 else None
                results.append(result)
            write_json(out / "summary.json", {"status": status, "n_completed_folds": len(metrics), "results": results})
            write_json(out / "run_status.json", {"status": status, "n_completed_folds": len(metrics)})
            return out, metrics
        except Exception as error:
            write_json(out / "run_status.json", {"status": "failed", "error": repr(error)})
            raise

    display(selection_pipeline())


## R01 — Kiểm tra S2

In [5]:
if RUN_TRAIN:
    CONFIG = make_config("r01", "quick")
    RUN_DIR, metrics = run_selection(CONFIG, subjects=["S2"])
    assert len(metrics) == 2
    display(metrics)
    display(read_csv(RUN_DIR / "warnings.csv"))


## R02 — LOSO đủ 15 người

Giữ nguyên grid và pipeline của R01, không điều chỉnh theo điểm test S2.

In [6]:
if RUN_TRAIN:
    CONFIG_R02 = make_config("r02", "main")
    RUN_DIR_R02, metrics_r02 = run_selection(CONFIG_R02)
    assert len(metrics_r02) == 30
    assert not metrics_r02.duplicated(["task", "model", "subject_id"]).any()
    for task in ["binary", "multiclass"]:
        assert set(metrics_r02.loc[metrics_r02.task.eq(task), "subject_id"]) == set(SUBJECTS)
    display(metrics_r02)
    summary_r02 = metrics_r02.groupby("task")[["macro_f1", "balanced_accuracy"]].agg(["mean", "std"])
    display(summary_r02)
    display(read_csv(RUN_DIR_R02 / "warnings.csv"))


## So sánh với LR đủ 68 đặc trưng

In [7]:
if RUN_TRAIN:
    reference_metrics = read_csv(REFERENCE_DIR / "metrics_by_subject.csv")
    comparison = metrics_r02.merge(reference_metrics, on=["task", "subject_id"],
                                  suffixes=("_selected", "_reference"), validate="one_to_one")
    assert len(comparison) == 30 and comparison.n_test_selected.eq(comparison.n_test_reference).all()
    reference_predictions = read_csv(REFERENCE_DIR / "predictions.csv")
    current_predictions = read_csv(RUN_DIR_R02 / "predictions.csv")
    paired = current_predictions.merge(reference_predictions, on=["task", "subject_id", "window_id"],
                                       suffixes=("_current", "_reference"), validate="one_to_one")
    assert len(paired) == 2 * len(df) and paired.y_true_current.eq(paired.y_true_reference).all()
    for metric in ["macro_f1", "balanced_accuracy"]:
        comparison[metric + "_delta"] = comparison[metric + "_selected"] - comparison[metric + "_reference"]
    display(comparison)
    display(comparison.groupby("task")[["macro_f1_delta", "balanced_accuracy_delta"]].agg(["mean", "std"]))
    comparison.to_csv(RUN_DIR_R02 / "comparison_by_subject.csv", index=False)


## Biểu đồ kết quả

In [8]:
if RUN_TRAIN:
    RUN_DIR_VIS = ROOT / "outputs/experiments/EXP-016/clean_v1/r02_main"
    REFERENCE_VIS = ROOT / "experiments/01_baselines/EXP-002_LR"
    MODEL_LABEL = "LR selected"
    REFERENCE_LABEL = "LR mặc định"
    FIGURES_VIS = RUN_DIR_VIS / "figures"

    for name in ["metrics_by_subject.csv", "predictions.csv"]:
        if not (RUN_DIR_VIS / name).is_file():
            raise FileNotFoundError(f"Thiếu kết quả: {RUN_DIR_VIS / name}")

    metrics_vis = read_csv(RUN_DIR_VIS / "metrics_by_subject.csv")
    predictions_vis = read_csv(RUN_DIR_VIS / "predictions.csv")
    assert not metrics_vis.duplicated(["task", "model", "subject_id"]).any()
    assert not predictions_vis.duplicated(["task", "model", "subject_id", "window_id"]).any()
    assert metrics_vis.model.nunique() == 1
    reference_vis = None
    if REFERENCE_VIS is not None:
        reference_vis = read_csv(REFERENCE_VIS / "metrics_by_subject.csv")
        paired_vis = metrics_vis.merge(reference_vis, on=["task", "subject_id"],
                                       suffixes=("_current", "_reference"), validate="one_to_one")
        assert len(paired_vis) == len(metrics_vis)
        assert paired_vis.n_test_current.eq(paired_vis.n_test_reference).all()

    FIGURES_VIS.mkdir(parents=True, exist_ok=True)
    plt.switch_backend("Agg")
    def show_figure(fig, filename):
        fig.tight_layout()
        target = FIGURES_VIS / filename
        fig.savefig(target, dpi=150, bbox_inches="tight")
        plt.close(fig)
        display(Image(filename=str(target)))

    summary_vis = metrics_vis.groupby("task")[["macro_f1", "balanced_accuracy"]].agg(["mean", "std"])
    display(summary_vis)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for ax, metric in zip(axes, ["macro_f1", "balanced_accuracy"]):
        means = summary_vis[(metric, "mean")]
        deviations = summary_vis[(metric, "std")].fillna(0)
        ax.bar(means.index, means, yerr=deviations, capsize=5, color=["#2c7fb8", "#41ab5d"][:len(means)])
        ax.set(title=metric.replace("_", " ").title(), ylim=(0, 1.05), ylabel="Điểm")
        ax.grid(axis="y", alpha=0.2)
        for x, mean in enumerate(means):
            ax.text(x, mean + 0.02, f"{mean:.3f}", ha="center")
    fig.suptitle(f"EXP-016 — {MODEL_LABEL}")
    show_figure(fig, "summary_metrics.png")

    for task in ["binary", "multiclass"]:
        part = metrics_vis.loc[metrics_vis.task.eq(task)].copy()
        if part.empty:
            continue
        subjects = sorted(part.subject_id.unique(), key=lambda s: int(s[1:]))
        part = part.set_index("subject_id").reindex(subjects)
        fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
        for ax, metric in zip(axes, ["macro_f1", "balanced_accuracy"]):
            scores = pd.DataFrame({MODEL_LABEL: part[metric]})
            if reference_vis is not None:
                ref = reference_vis.loc[reference_vis.task.eq(task)].set_index("subject_id").reindex(subjects)
                scores[REFERENCE_LABEL] = ref[metric]
            scores.plot.bar(ax=ax, ylim=(0, 1.05), rot=0)
            ax.set_ylabel(metric.replace("_", " ").title())
            ax.grid(axis="y", alpha=0.2)
        axes[0].set_title(f"EXP-016 — {task}: kết quả theo người test")
        axes[1].set_xlabel("Người test")
        show_figure(fig, f"{task}_subject_metrics.png")

        macro_scores = pd.DataFrame({MODEL_LABEL: part["macro_f1"]})
        if reference_vis is not None:
            macro_scores[REFERENCE_LABEL] = ref["macro_f1"]
        ax_macro = macro_scores.plot.bar(figsize=(10, 4), ylim=(0, 1.05), rot=0)
        ax_macro.set(title=f"EXP-016 — {task}", ylabel="Macro-F1", xlabel="Người test")
        ax_macro.figure.tight_layout()
        ax_macro.figure.savefig(FIGURES_VIS / f"{task}_macro_f1.png", dpi=150)
        plt.close(ax_macro.figure)


        pred = predictions_vis.loc[predictions_vis.task.eq(task)]
        assert len(pred) == int(part.n_test.sum())
        labels = [0, 1] if task == "binary" else [1, 2, 3]
        names = ["Non-stress", "Stress"] if task == "binary" else ["Baseline", "Stress", "Amusement"]
        cm = confusion_matrix(pred.y_true, pred.y_pred, labels=labels)
        row_sums = cm.sum(axis=1, keepdims=True)
        normalized = np.divide(cm, row_sums, out=np.zeros_like(cm, dtype=float), where=row_sums != 0)
        fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
        ConfusionMatrixDisplay(cm, display_labels=names).plot(ax=axes[0], cmap="Blues", colorbar=False, values_format="d")
        ConfusionMatrixDisplay(normalized, display_labels=names).plot(ax=axes[1], cmap="Blues", colorbar=False, values_format=".2f")
        axes[0].set_title("Số cửa sổ")
        axes[1].set_title("Tỷ lệ theo nhãn thật")
        fig.suptitle(f"EXP-016 — {task}: confusion matrix")
        show_figure(fig, f"{task}_confusion_matrix.png")

    if reference_vis is not None:
        fig, axes = plt.subplots(2, 1, figsize=(11, 6))
        for ax, task in zip(axes, ["binary", "multiclass"]):
            part = paired_vis.loc[paired_vis.task.eq(task)].copy()
            part["subject_number"] = part.subject_id.str[1:].astype(int)
            part = part.sort_values("subject_number")
            delta = part.macro_f1_current - part.macro_f1_reference
            ax.bar(part.subject_id, delta, color=["#2c7fb8" if d >= 0 else "#e6550d" for d in delta])
            ax.axhline(0, color="black", linewidth=0.8)
            ax.set(title=task, ylabel="Chênh lệch Macro-F1")
            ax.grid(axis="y", alpha=0.2)
        fig.suptitle(f"{MODEL_LABEL} so với {REFERENCE_LABEL}")
        show_figure(fig, "comparison_delta_macro_f1.png")

    print("Ảnh được lưu tại:", FIGURES_VIS)
    if metrics_vis.subject_id.nunique() == 1:
        print("Lượt kiểm tra một người; chưa dùng để kết luận cho đủ 15 người.")
    else:
        print("Cột sai số ở bảng tổng hợp là độ lệch chuẩn giữa các người.")
    print("Confusion matrix gộp cửa sổ; bảng tổng hợp tính trung bình theo người.")


## Số đặc trưng được chọn trong từng fold

In [9]:
if RUN_TRAIN:
    fig, axes = plt.subplots(2, 1, figsize=(11, 6))
    for ax, task in zip(axes, ["binary", "multiclass"]):
        part = metrics_r02.loc[metrics_r02.task.eq(task)].set_index("subject_id").reindex(SUBJECTS)
        ax.bar(part.index, part.selected_k, color="#2c7fb8")
        ax.set(title=task, ylabel="Số đặc trưng", ylim=(0, 72), yticks=[0, 16, 32, 48, 68])
        ax.grid(axis="y", alpha=0.2)
    show_figure(fig, "selected_k_by_subject.png")


## Kết quả và nhận xét R02

Đã hoàn thành 30 fold cho hai bài toán. Không ghi nhận cảnh báo.

- binary: Macro-F1 = 0.878943 ± 0.148313; Balanced Accuracy = 0.888401 ± 0.135539.
  So với LR đủ 68 đặc trưng: delta Macro-F1 = +0.068674; tăng ở 10 người, giảm ở 3 người và gần như bằng ở 2 người.
- multiclass: Macro-F1 = 0.612709 ± 0.176236; Balanced Accuracy = 0.658928 ± 0.142906.
  So với LR đủ 68 đặc trưng: delta Macro-F1 = +0.023318; tăng ở 11 người, giảm ở 4 người và gần như bằng ở 0 người.

Cả hai bài toán tăng điểm trung bình nhưng không cải thiện trên mọi người. Số đặc trưng được chọn khác nhau giữa các fold; ở binary có một fold giữ đủ 68 đặc trưng.
Không dùng kết quả outer test để chọn lại k hoặc sửa grid.

## Giới hạn

Chưa kiểm định ý nghĩa thống kê và chưa có tập test ngoài WESAD. Không coi điểm tăng trung bình là bằng chứng hiệu quả triển khai thực tế.

## Quyết định tiếp theo

Giữ kết quả này để tổng hợp cùng LR mặc định và LR cân bằng lớp. Việc chọn cấu hình cuối cần theo tiêu chí chung của dự án.

In [10]:
saved = pd.read_csv(FOLDER / "results.csv")
assert len(saved) == 30
display(saved.groupby("task")[["macro_f1", "balanced_accuracy"]].agg(["mean", "std"]))

            macro_f1           balanced_accuracy          
                mean       std              mean       std
task                                                      
binary      0.878943  0.148313          0.888401  0.135539
multiclass  0.612709  0.176236          0.658928  0.142906
